# Workshop 1 - Data Preprocessing and Supervised Learning


## Contexto y enfoque

Este trabajo se apoyará en los notebooks de referencia `MIAX ML - 01 Data Preprocessing.ipynb` y `MIAX ML - 03 Linear Regression.ipynb`.

El dataset base serán `navs.pickle`.
Se prioriza simplicidad, claridad y justificación explícita de cada decisión metodológica.

## 1. Objetivo y tesis financiera

### Hipótesis financiera
Como gestores de un fondo de fondos, planteamos la hipótesis de que **los fondos con mejor comportamiento histórico reciente, especialmente los expuestos a Asia, tenderán a mantener un desempeño relativo superior en el siguiente horizonte de inversión**, siempre que se combine este sesgo con diversificación para controlar riesgo idiosincrático.

Esta tesis se apoya en dos ideas conocidas en finanzas:
1. **Persistencia parcial del momentum**: activos/fondos con buen desempeño reciente pueden seguir mostrando fortaleza durante un tiempo.
2. **Sesgo temático con diversificación**: sobreponderar una región con tesis macro (Asia) puede capturar una oportunidad estructural, pero debe combinarse con otros fondos para evitar concentración excesiva.

### Justificación económica y evidencia previa

### Objetivo principal

> Nota: aunque una regresión de retorno futuro es viable, en este caso la clasificación es más alineada con la decisión real (supera/no supera benchmark).

Esto convierte el problema en clasificación binaria de “outperformers”.

### Horizonte temporal de predicción
Se evaluarán dos horizontes candidatos:
- **1 mes**: más observaciones y capacidad de reacción, pero mayor ruido.
- **1 año**: más alineado con construcción estratégica de cartera (fondo de fondos), pero menor tamaño muestral y menor reactividad.

Como hipótesis base para esta práctica, se prioriza **1 año** por coherencia con el mandato de asignación estratégica, y se deja **1 mes** como análisis de sensibilidad.

### Restricciones y alcance de datos

### Prevención de look-ahead 

### Métricas candidatas y criterio de evaluación


## 2. Carga y alineación temporal de datos

Esta sección no busca todavía “mejorar” los datos, sino **hacerlos comparables en el tiempo** y dejar un punto de partida reproducible para el resto del pipeline.  
La decisión central es separar claramente tres capas:

1. **Estructura** (qué objetos y columnas tenemos),
2. **Tiempo** (cómo representamos fechas y cronología),
3. **Persistencia** (qué versión del dato guardamos para reutilizar).

El motivo es metodológico: si se mezclan decisiones de limpieza avanzada (fase 4) con alineación temporal (fase 2), se pierde trazabilidad y luego es más difícil justificar por qué una métrica cambió.

Además, en un problema financiero, la consistencia temporal es crítica: antes de modelar hay que garantizar que las series están en un marco común y que no hay artefactos de indexado/duplicación.

Posibles críticas
- Aún no se fija un criterio definitivo de exclusión por historial mínimo de fondos.
- El pipeline prioriza reproducibilidad sobre eficiencia extrema de memoria.
- Parte de las decisiones (por ejemplo, panel diario completo) pueden ser costosas computacionalmente.


#### 2.1 Configuración y librerías

La configuración se centraliza al inicio (`PICKLE_PATH`, carpetas de salida, frecuencia semanal objetivo, límite de `ffill`) para evitar “constantes ocultas” dentro de celdas posteriores.  
La razón práctica es doble:

- **Reproducibilidad**: cambiar una ruta o parámetro en un único sitio.
- **Auditabilidad**: dejar explícitas decisiones de diseño (por ejemplo `W-FRI`, `FFILL_LIMIT=5`) antes de ejecutar transformaciones.

También se importan librerías de descarga y parseo robusto (`requests`, `zipfile`, `io`) porque se decidió evitar dependencias frágiles en la obtención de factores `panda_datareader`.

Posibles críticas
- Hay imports que podrían no usarse en todas las ejecuciones (sobrecarga visual).
- Algunos parámetros están marcados como “pendientes”, lo que puede dar sensación de diseño no cerrado.
- Conviene separar configuración NAVs vs factores para mayor limpieza modular.

In [ ]:
from pathlib import Path
import warnings
import os
import numpy as np
import pandas as pd
import pandas_datareader.data as web
from collections import Counter
import io
import zipfile
import requests

# =========================
# Configuración
# =========================
PICKLE_PATH = Path("DOCS_CLASE") / "MachineLearning" / "dataset" / "navs.pickle"
OUTPUT_DIR = Path("data") / "processed"  #REVISAR REVISAR REVSIAR
OUTPUT_PARQUET = OUTPUT_DIR / "navs_weekly_aligned.parquet" # REVISAR REVISAR

#QUITAR SI NO SE USA EN ESTA PARTE
WEEK_FREQ = "W-FRI"          # criterio acordado 
FFILL_LIMIT = 5               # máximo de periodos consecutivos a rellenar
MIN_WEEKS_REQUIRED = None     # pendiente de decisión (no filtra en esta fase)



#### 2.2 Carga y visualización inicial del `.pickle`

Aquí se valida primero la **estructura global del activo principal** (`navs_dict`) antes de transformar nada.  
La decisión de revisar `nº fondos`, `shape` agregado, columnas comunes y ejemplos concretos responde a una pregunta clave:  
**¿estamos tratando con un universo homogéneo o con múltiples excepciones estructurales?**

Se imprime una muestra de fondos para verificar que:
- el índice temporal existe,
- las columnas esperadas (`isin`, `allfunds_id`, `nav`, `name`) son consistentes,
- y el contenido es coherente con el contexto financiero.

Esto reduce riesgo de errores silenciosos en pasos posteriores (por ejemplo, asumir `date` como columna cuando viene en índice).

 Posibles críticas
- El `shape` agregado puede inducir falsa sensación de completitud (no refleja huecos temporales).
- Mostrar solo 3 fondos puede no capturar outliers estructurales.
- Faltaría un control cuantitativo de fondos con estructura distinta del patrón dominante.

In [3]:
# 1) Cargar archivo
navs_dict = pd.read_pickle(PICKLE_PATH)

# Validación básica
if not isinstance(navs_dict, dict):
    raise TypeError(f"Se esperaba un dict y se recibió: {type(navs_dict)}")

print("Archivo cargado correctamente")
print(f"Tipo de objeto: {type(navs_dict).__name__}")
print(f"Cantidad de fondos: {len(navs_dict):,}")

# 2) Métricas globales: filas totales, columnas comunes y shape global lógico
n_fondos = len(navs_dict)
total_filas = 0
common_cols = None
shapes = []
non_df_funds = 0

for fund_id, df in navs_dict.items():
    if not isinstance(df, pd.DataFrame):
        non_df_funds += 1
        continue

    shapes.append(df.shape)
    total_filas += df.shape[0]

    cols = set(df.columns)
    if common_cols is None:
        common_cols = cols
    else:
        common_cols = common_cols.intersection(cols)

common_cols = sorted(common_cols) if common_cols is not None else []
shape_global = (total_filas, len(common_cols))  # shape lógico agregado

print(f"Shape global (agregado lógico): {shape_global}")
print(f"Columnas comunes ({len(common_cols)}): {common_cols}")
if non_df_funds > 0:
    print(f"Advertencia: {non_df_funds} fondos no son DataFrame")

# 3) Distribución rápida de shapes (top 5)
shape_counts = Counter(shapes)
print("\nTop 5 shapes más frecuentes:")
for shp, cnt in shape_counts.most_common(5):
    print(f"  {shp}: {cnt} fondos")

# 4) Mostrar algunos ejemplos de fondos
sample_ids = list(navs_dict.keys())[:3]
print(f"\nEjemplo de fondos (primeros {len(sample_ids)}):")
for fid in sample_ids:
    obj = navs_dict[fid]
    print(f"\n--- Fondo {fid} ---")
    if isinstance(obj, pd.DataFrame):
        print(f"shape: {obj.shape}")
        print(f"columnas: {obj.columns.tolist()}")
        display(obj.head(5))
    else:
        print(f"No es DataFrame. Tipo: {type(obj)}")

Archivo cargado correctamente
Tipo de objeto: dict
Cantidad de fondos: 24,822
Shape global (agregado lógico): (29455509, 4)
Columnas comunes (4): ['allfunds_id', 'isin', 'name', 'nav']

Top 5 shapes más frecuentes:
  (1387, 4): 977 fondos
  (1383, 4): 819 fondos
  (1384, 4): 695 fondos
  (1385, 4): 689 fondos
  (1425, 4): 576 fondos

Ejemplo de fondos (primeros 3):

--- Fondo 90 ---
shape: (1387, 4)
columnas: ['isin', 'allfunds_id', 'nav', 'name']


,isin,allfunds_id,nav,name
date,,,,
2016-01-05,LU0171310443,90,16.47,"BGF WORLD TECHNOLOGY ""A2"" (EUR)"
2016-01-06,LU0171310443,90,16.19,"BGF WORLD TECHNOLOGY ""A2"" (EUR)"
2016-01-07,LU0171310443,90,15.68,"BGF WORLD TECHNOLOGY ""A2"" (EUR)"
2016-01-08,LU0171310443,90,15.59,"BGF WORLD TECHNOLOGY ""A2"" (EUR)"
2016-01-11,LU0171310443,90,15.26,"BGF WORLD TECHNOLOGY ""A2"" (EUR)"



--- Fondo 541 ---
shape: (1373, 4)
columnas: ['isin', 'allfunds_id', 'nav', 'name']


,isin,allfunds_id,nav,name
date,,,,
2016-01-05,LU0248272758,541,27.54,"BGF INDIA ""A2"""
2016-01-06,LU0248272758,541,27.50,"BGF INDIA ""A2"""
2016-01-07,LU0248272758,541,26.83,"BGF INDIA ""A2"""
2016-01-08,LU0248272758,541,27.21,"BGF INDIA ""A2"""
2016-01-11,LU0248272758,541,26.98,"BGF INDIA ""A2"""



--- Fondo 909 ---
shape: (1299, 4)
columnas: ['isin', 'allfunds_id', 'nav', 'name']


,isin,allfunds_id,nav,name
date,,,,
2016-05-11,LU1408527916,909,10.00,"BGF EMERGING MARKETS ""A"" (GBPHDG) D"
2016-05-12,LU1408527916,909,10.03,"BGF EMERGING MARKETS ""A"" (GBPHDG) D"
2016-05-13,LU1408527916,909,10.06,"BGF EMERGING MARKETS ""A"" (GBPHDG) D"
2016-05-17,LU1408527916,909,10.07,"BGF EMERGING MARKETS ""A"" (GBPHDG) D"
2016-05-18,LU1408527916,909,10.07,"BGF EMERGING MARKETS ""A"" (GBPHDG) D"


#### 2.3 Unificación diaria del universo en un único dataset

La decisión de transformar el diccionario de fondos en un **dataset largo diario** busca simplificar operaciones posteriores (merge temporal, controles de cobertura, joins con factores).  
Se convierte el índice a columna `date`, se estandarizan tipos y se resuelven duplicados por (`date`, `isin`) conservando la última observación, priorizando consistencia de clave temporal.

Se construye un panel diario completo `date x isin` para disponer de una matriz explícita de disponibilidad de NAV (observado vs faltante).  
Este enfoque favorece la transparencia de faltantes frente a enfoques donde los huecos quedan implícitos.

Guardar en `parquet` responde a una necesidad operativa: el CSV completo era muy pesado para trabajo y versionado.

Posibles críticas
- El producto cartesiano `date x isin` puede ser excesivo en memoria y disco.
- Conservar la última fila en duplicados asume que el orden original es fiable.
- Un panel completo diario puede no ser necesario si luego se trabajará en frecuencia semanal.

Decisión de estructura: identificador único del fondo

Durante la construcción del dataset diario se detectó redundancia entre `fund_id` (clave del diccionario original) y `allfunds_id` (campo del propio dataset).  
En la práctica, ambas columnas representan el mismo identificador de fondo en las observaciones válidas.

Para reducir dimensionalidad y evitar duplicación semántica, se elimina `fund_id` y se conserva `allfunds_id` como identificador canónico, por ser el campo nativo del dato fuente y facilitar trazabilidad con el esquema original.

In [4]:
# Pipeline: navs_dict -> panel diario completo -> parquet
# Requiere navs_dict ya cargado en memoria

if "navs_dict" not in globals():
    raise ValueError("No existe `navs_dict` en memoria. Ejecuta primero la celda de carga.")

print("[1/6] Validando entrada...")
if not isinstance(navs_dict, dict):
    raise TypeError(f"`navs_dict` debe ser dict, recibido: {type(navs_dict)}")
print(f"    Fondos en diccionario: {len(navs_dict):,}")

print("[2/6] Aplanando a formato largo...")
rows = []

for fund_id, df in navs_dict.items():
    if not isinstance(df, pd.DataFrame):
        continue

    tmp = df.copy().reset_index()  # date está en índice
    tmp.columns = [str(c).strip().lower() for c in tmp.columns]

    # asegurar date
    if "date" not in tmp.columns:
        if "index" in tmp.columns:
            tmp = tmp.rename(columns={"index": "date"})
        else:
            tmp = tmp.rename(columns={tmp.columns[0]: "date"})

    # asegurar nav
    if "nav" not in tmp.columns:
        continue

    tmp["fund_id"] = fund_id
    # Evitamos duplicar IDs: conservamos allfunds_id como identificador canónico del fondo
    keep = ["date", "allfunds_id", "isin", "name", "nav"]
    keep = [c for c in keep if c in tmp.columns]
    tmp = tmp[keep]

    rows.append(tmp)

if not rows:
    raise RuntimeError("No se pudieron extraer fondos válidos con date/nav.")

daily_long_raw = pd.concat(rows, ignore_index=True)
print(f"    Filas concatenadas: {len(daily_long_raw):,}")

print("[3/6] Limpieza mínima de fecha y duplicados...")
daily_long_raw["date"] = pd.to_datetime(daily_long_raw["date"], errors="coerce", utc=True).dt.tz_localize(None)
daily_long_raw["nav"] = pd.to_numeric(daily_long_raw["nav"], errors="coerce")
daily_long_raw = daily_long_raw.dropna(subset=["date"]).copy()

if "isin" not in daily_long_raw.columns:
    raise RuntimeError("No existe columna 'isin'; no se puede construir panel por ISIN.")

daily_long_raw = (
    daily_long_raw
    .sort_values(["isin", "date"], kind="mergesort")
    .drop_duplicates(subset=["date", "isin"], keep="last")
    .reset_index(drop=True)
)

print("[4/6] Creando panel diario completo (date x isin)...")
global_min = daily_long_raw["date"].min()
global_max = daily_long_raw["date"].max()
all_days = pd.date_range(global_min, global_max, freq="D")
all_isins = daily_long_raw["isin"].dropna().drop_duplicates().sort_values()

calendar_df = pd.DataFrame({"date": all_days})
isins_df = pd.DataFrame({"isin": all_isins.values})
calendar_df["k"] = 1
isins_df["k"] = 1
daily_panel = calendar_df.merge(isins_df, on="k", how="inner").drop(columns="k")

obs = daily_long_raw[["date", "isin", "nav"]]
daily_panel = daily_panel.merge(obs, on=["date", "isin"], how="left")

meta_cols = [c for c in ["allfunds_id", "name"] if c in daily_long_raw.columns]
if meta_cols:
    meta_by_isin = (
        daily_long_raw
        .sort_values(["isin", "date"])
        .drop_duplicates(subset=["isin"], keep="last")[["isin"] + meta_cols]
    )
    daily_panel = daily_panel.merge(meta_by_isin, on="isin", how="left")

daily_panel = daily_panel.sort_values(["date", "isin"]).reset_index(drop=True)

print(f"    Rango global: {global_min.date()} -> {global_max.date()}")
print(f"    Filas panel: {len(daily_panel):,}")

print("[5/6] Guardando parquet...")
OUT_DIR = Path("data") / "daily_universe"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PARQUET = OUT_DIR / "panel_diario_universo.parquet"
if OUT_PARQUET.exists():
    OUT_PARQUET.unlink()

daily_panel.to_parquet(OUT_PARQUET, index=False)

print(f"    Parquet guardado en: {OUT_PARQUET.as_posix()}")

print("[6/6] Vista previa")
display(daily_panel.head(10))

[1/6] Validando entrada...
    Fondos en diccionario: 24,822
[2/6] Aplanando a formato largo...
    Filas concatenadas: 29,455,509
[3/6] Limpieza mínima de fecha y duplicados...
[4/6] Creando panel diario completo (date x isin)...
    Rango global: 2016-01-05 -> 2021-07-16
    Filas panel: 50,134,380
[5/6] Guardando parquet...
    Parquet guardado en: data/daily_universe/panel_diario_universo.parquet
[6/6] Vista previa


,date,isin,nav,allfunds_id,name
0,2016-01-05,AT0000494893,297.13,60265,"ERSTE STOCK ISTANBUL ""VT"" (EUR)"
1,2016-01-05,AT0000495304,142.24,21156,"RAIFFEISEN TOP DIVIDEND EQ ""R-VTA"" ACC"
2,2016-01-05,AT0000497268,215.88,21157,"RAIFFEISEN EMERGING M ""R"" (EUR) A"
3,2016-01-05,AT0000607270,177.58,817028,"RAIFFEISEN 304 EURO COR ""I-VTA"" ACC"
4,2016-01-05,AT0000673165,484.11,60256,"ERSTE STOCK BIOTEC ""VTA"" (EUR)"
5,2016-01-05,AT0000673181,74.41,60271,"ERSTE STOCK EUROPE EMERGING ""VT"" (EUR)"
6,2016-01-05,AT0000673199,172.47,60268,"ERSTE BOND DANUBIA ""VT"" EUR"
7,2016-01-05,AT0000673306,181.43,173258,"ERSTE BOND EM GOVERNMENT ""VT"" (EUR)"
8,2016-01-05,AT0000704333,234.64,60264,"ERSTE STOCK ISTANBUL ""A"" (EUR) D"
9,2016-01-05,AT0000712534,190.69,817008,"RAIFFEISEN EURO CORPO ""R"" (EUR) A"


In [ ]:

archivo = Path(OUT_PARQUET)  # revisa si es .parquet o .parqeut

if archivo.exists():
    size_bytes = archivo.stat().st_size
    size_kb = size_bytes / 1024
    size_mb = size_bytes / (1024 ** 2)
    size_gb = size_bytes / (1024 ** 3)

    print(f"Archivo: {archivo}")
    print(f"Tamaño: {size_bytes:,} bytes")
    print(f"Tamaño: {size_kb:,.2f} KB")
    print(f"Tamaño: {size_mb:,.2f} MB")
    print(f"Tamaño: {size_gb:,.4f} GB")
else:
    print(f"No se encontró el archivo: {archivo.resolve()}")

Archivo: data\daily_universe\panel_diario_universo.parquet
Tamaño: 1,020,951,389 bytes
Tamaño: 997,022.84 KB
Tamaño: 973.66 MB
Tamaño: 0.9508 GB


#### 2.4 Carga de factores Fama-French (Asia ex Japan) con flujo robusto

Se decide descargar factores directamente desde los ZIP oficiales de Ken French en lugar de depender de `pandas_datareader`. El motivo principal es de **robustez y mantenimiento**: evitar warnings deprecados internos y reducir dependencia de implementaciones intermedias que pueden cambiar.

La normalización posterior (`date` a `datetime`, orden, deduplicación, conversión numérica) busca dejar factores listos para cruce temporal con NAVs sin ambigüedades de parseo.

Guardar una copia local en `data/factors` responde a reproducibilidad: una vez descargado, el pipeline no depende de red en cada ejecución.

Posibles críticas
- El parseo manual de ZIP puede ser más frágil si la estructura del proveedor cambia.
- Faltan checks automáticos de integridad (por ejemplo, columnas esperadas exactas por dataset).
- Podría añadirse versionado de fecha de descarga para trazabilidad histórica.

In [12]:
# =========================================================
# 2.x Carga de factores (sin pandas_datareader)
# Descarga directa desde Ken French (ZIP) + guardado en CSV
# =========================================================

print("[1/7] Configurando carpeta de salida...")
FACTORS_DIR = Path("data") / "factors"
FACTORS_DIR.mkdir(parents=True, exist_ok=True)
print(f"    Carpeta: {FACTORS_DIR.as_posix()}")

print("[2/7] Definiendo función de descarga/parseo robusta...")

def load_ken_french_daily_zip(url: str) -> pd.DataFrame:
    # Descargar ZIP
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    # Leer archivo interno txt/csv
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        inner_files = [n for n in zf.namelist() if n.lower().endswith((".txt", ".csv"))]
        if not inner_files:
            raise RuntimeError(f"No se encontró TXT/CSV dentro del ZIP: {url}")
        inner_name = inner_files[0]
        lines = zf.read(inner_name).decode("latin1", errors="ignore").splitlines()

    # Buscar inicio de tabla diaria: primera línea con fecha YYYYMMDD
    start_data = None
    for i, line in enumerate(lines):
        s = line.strip()
        if len(s) >= 8 and s[:8].isdigit():
            # normalmente el header está justo arriba
            start_data = max(i - 1, 0)
            break
    if start_data is None:
        raise RuntimeError(f"No se encontró bloque de datos en: {url}")

    # Cortar bloque útil (hasta notas/metadatos del final)
    block = []
    for line in lines[start_data:]:
        s = line.strip()
        if s == "":
            # después de arrancar y tener varias líneas, vacío suele marcar fin de bloque
            if len(block) > 5:
                break
            continue
        block.append(line)

    if len(block) < 2:
        raise RuntimeError(f"Bloque de datos insuficiente en: {url}")

    # Parsear tabla
    df = pd.read_csv(io.StringIO("\n".join(block)))

    # Normalizar columna fecha
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "date"})
    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")

    # Limpieza mínima estructural
    df = (
        df.dropna(subset=["date"])
          .sort_values("date")
          .drop_duplicates(subset=["date"], keep="last")
          .reset_index(drop=True)
    )

    # Convertir factores a numérico
    for c in df.columns:
        if c != "date":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


print("[3/7] Descargando factores Asia ex Japan desde Ken French...")
URL_3F = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Asia_Pacific_ex_Japan_3_Factors_Daily_CSV.zip"
URL_MOM = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/Asia_Pacific_ex_Japan_MOM_Factor_Daily_CSV.zip"

ff_asia = load_ken_french_daily_zip(URL_3F)
mom_asia = load_ken_french_daily_zip(URL_MOM)

print("[4/7] Control de estructura descargada...")
print(f"    FF filas: {len(ff_asia):,} | columnas: {ff_asia.columns.tolist()}")
print(f"    MOM filas: {len(mom_asia):,} | columnas: {mom_asia.columns.tolist()}")
print(f"    FF rango: {ff_asia['date'].min().date()} -> {ff_asia['date'].max().date()}")
print(f"    MOM rango: {mom_asia['date'].min().date()} -> {mom_asia['date'].max().date()}")

print("[5/7] Guardando CSV locales (reemplazo si existen)...")
FF_CSV = FACTORS_DIR / "ff_asia_pacific_3factors_daily.csv"
MOM_CSV = FACTORS_DIR / "mom_asia_pacific_daily.csv"

if FF_CSV.exists():
    FF_CSV.unlink()
if MOM_CSV.exists():
    MOM_CSV.unlink()

ff_asia.to_csv(FF_CSV, index=False, encoding="utf-8")
mom_asia.to_csv(MOM_CSV, index=False, encoding="utf-8")

print(f"    Guardado: {FF_CSV.as_posix()}")
print(f"    Guardado: {MOM_CSV.as_posix()}")

print("[6/7] Preview")
display(ff_asia.head(5))
display(mom_asia.head(5))

print("[7/7] Factores listos para cruce temporal con NAVs.")

[1/7] Configurando carpeta de salida...
    Carpeta: data/factors
[2/7] Definiendo función de descarga/parseo robusta...
[3/7] Descargando factores Asia ex Japan desde Ken French...
[4/7] Control de estructura descargada...
    FF filas: 9,285 | columnas: ['date', 'Mkt-RF', 'SMB', 'HML', 'RF']
    MOM filas: 9,197 | columnas: ['date', 'WML']
    FF rango: 1990-07-02 -> 2026-01-30
    MOM rango: 1990-11-01 -> 2026-01-30
[5/7] Guardando CSV locales (reemplazo si existen)...
    Guardado: data/factors/ff_asia_pacific_3factors_daily.csv
    Guardado: data/factors/mom_asia_pacific_daily.csv
[6/7] Preview


,date,Mkt-RF,SMB,HML,RF
0,1990-07-02,0.40,-0.09,0.60,0.03
1,1990-07-03,0.86,-0.64,0.30,0.03
2,1990-07-04,1.28,-0.29,-0.30,0.03
3,1990-07-05,0.20,-0.57,-0.34,0.03
4,1990-07-06,-0.07,-0.16,0.42,0.03


,date,WML
0,1990-11-01,0.72
1,1990-11-02,0.96
2,1990-11-05,-1.41
3,1990-11-06,-0.13
4,1990-11-07,0.01


[7/7] Factores listos para cruce temporal con NAVs.


#### 2.5 Unificación de factores en un único dataset

Unificar `Mkt-RF`, `SMB`, `HML`, `RF` con `WML` en una sola tabla reduce complejidad en etapas de modelado y evita joins repetidos.  
La decisión de usar `inner join` por `date` es deliberada: solo se conservan fechas donde todos los factores relevantes existen, garantizando una base consistente para features.

Esto implica que el inicio temporal queda determinado por el factor con menor historial (`WML`), lo cual en este caso es aceptable porque no se requiere explotar periodos anteriores para el análisis actual.

El resultado (`factores_unificados.csv`) actúa como fuente canónica de factores para el resto del notebook.

Posibles críticas
- `inner join` sacrifica historial de factores clásicos previo a `WML`.
- No se analiza si faltan fechas puntuales en algún factor dentro del rango común.
- Si el objetivo final usa horizontes largos, convendría evaluar impacto de recortar muestra al inicio.

In [13]:
#Unificación de factores (3 factores + momentum WML)

print("[1/6] Cargando CSV de factores...")
FACTORS_DIR = Path("data") / "factors"
ff_path = FACTORS_DIR / "ff_asia_pacific_3factors_daily.csv"
mom_path = FACTORS_DIR / "mom_asia_pacific_daily.csv"

ff = pd.read_csv(ff_path)
mom = pd.read_csv(mom_path)

print(f"    FF shape : {ff.shape}")
print(f"    MOM shape: {mom.shape}")

print("[2/6] Normalizando fechas y tipos...")
ff["date"] = pd.to_datetime(ff["date"], errors="coerce")
mom["date"] = pd.to_datetime(mom["date"], errors="coerce")

ff = ff.dropna(subset=["date"]).copy()
mom = mom.dropna(subset=["date"]).copy()

# Asegurar numérico en factores
for c in ["Mkt-RF", "SMB", "HML", "RF"]:
    if c in ff.columns:
        ff[c] = pd.to_numeric(ff[c], errors="coerce")

if "WML" in mom.columns:
    mom["WML"] = pd.to_numeric(mom["WML"], errors="coerce")
else:
    raise RuntimeError("No se encontró la columna 'WML' en mom_asia_pacific_daily.csv")

print("[3/6] Ordenando y quitando duplicados por fecha...")
ff = ff.sort_values("date").drop_duplicates(subset=["date"], keep="last").reset_index(drop=True)
mom = mom.sort_values("date").drop_duplicates(subset=["date"], keep="last").reset_index(drop=True)

print("[4/6] Uniendo datasets por fecha...")
# inner join => solo fechas comunes (arranca cuando existe WML)
factores_unificados = ff.merge(mom[["date", "WML"]], on="date", how="inner")
factores_unificados = factores_unificados.sort_values("date").reset_index(drop=True)

print("[5/6] Control de resultado...")
print(f"    Shape unificado: {factores_unificados.shape}")
print(f"    Columnas: {factores_unificados.columns.tolist()}")
print(f"    Rango: {factores_unificados['date'].min().date()} -> {factores_unificados['date'].max().date()}")

print("[6/6] Guardando CSV final...")
out_path = FACTORS_DIR / "factores_unificados.csv"
if out_path.exists():
    out_path.unlink()

factores_unificados.to_csv(out_path, index=False, encoding="utf-8")
print(f"    Guardado en: {out_path.as_posix()}")

display(factores_unificados.head(10))

[1/6] Cargando CSV de factores...
    FF shape : (9285, 5)
    MOM shape: (9197, 2)
[2/6] Normalizando fechas y tipos...
[3/6] Ordenando y quitando duplicados por fecha...
[4/6] Uniendo datasets por fecha...
[5/6] Control de resultado...
    Shape unificado: (9197, 6)
    Columnas: ['date', 'Mkt-RF', 'SMB', 'HML', 'RF', 'WML']
    Rango: 1990-11-01 -> 2026-01-30
[6/6] Guardando CSV final...
    Guardado en: data/factors/factores_unificados.csv


,date,Mkt-RF,SMB,HML,RF,WML
0,1990-11-01,-1.38,0.97,0.07,0.03,0.72
1,1990-11-02,-1.03,-0.05,-0.26,0.03,0.96
2,1990-11-05,1.20,-0.67,0.26,0.03,-1.41
3,1990-11-06,0.40,0.01,-0.30,0.03,-0.13
4,1990-11-07,-0.22,-0.51,-0.43,0.03,0.01
5,1990-11-08,-0.14,-0.32,1.05,0.03,0.03
6,1990-11-09,-0.56,0.70,-0.15,0.03,0.67
7,1990-11-12,0.44,-0.52,0.00,0.03,0.60
8,1990-11-13,0.87,-1.32,-0.19,0.03,0.42
9,1990-11-14,-0.29,-0.46,0.46,0.03,1.19


## 3. EDA (exploración de calidad)

En esta parte se revisará calidad de datos de forma inicial: tamaños, tipos, nulos y comportamientos básicos.
El objetivo es detectar problemas antes de limpiar y modelar.
. 

**Decisiones a justificar**
- Qué indicadores de calidad son prioritarios.
- Qué gráficos mínimos aportan más información.
- Qué umbrales preliminares se usarán para alertas.

**Entregable de la sección:** resumen de calidad con tablas y gráficos básicos de diagnóstico.

In [ ]:
# TODO: Revisar shape, dtypes y nulos
# TODO: Generar visualizaciones básicas de calidad de datos

## 4. Limpieza y preprocesado

Aquí se aplicarán reglas explícitas para tratar faltantes, duplicados y valores inconsistentes.
La intención es mejorar la calidad sin introducir sesgos innecesarios.

**Decisiones a justificar**
- Umbral para excluir fondos con demasiados faltantes.
- Método de imputación elegido (si aplica).
- Criterio para tratar valores no positivos y outliers.

**Entregable de la sección:** versión limpia del dataset con reglas documentadas.

In [ ]:
# TODO: Definir y aplicar reglas de limpieza
# TODO: Guardar tablas resumen antes/después de limpiar

## 5. Feature engineering

En este bloque se transformarán NAVs y factores en variables listas para modelado supervisado.
Se buscará un conjunto de features simple, interpretable y temporalmente correcto.

**Decisiones a justificar**
- Definición de retornos y transformaciones aplicadas.
- Variables predictoras elegidas (actuales y/o rezagadas).
- Definición final de `X` y `y`.

**Entregable de la sección:** matriz de features y variable objetivo preparadas para train/test.

In [ ]:
# TODO: Construir features (retornos, rezagos, variables seleccionadas)
# TODO: Definir variable objetivo para supervisado

## 6. Modelado supervisado

Aquí se entrenarán modelos supervisados simples para evaluar la señal predictiva de las features.
La prioridad será comparar de forma clara y con supuestos controlados.

**Decisiones a justificar**
- Modelo principal seleccionado y por qué.
- Configuración mínima de entrenamiento.
- Cómo se evita complejidad innecesaria.

**Entregable de la sección:** modelo entrenado y predicciones sobre el conjunto de test.

In [ ]:
# TODO: Crear split temporal train/test sin shuffle
# TODO: Entrenar modelo supervisado base y generar predicciones

## 7. Evaluación y visualización

Esta sección medirá el rendimiento del modelo con métricas adecuadas al objetivo elegido.
También se incluirán visualizaciones simples para interpretar los resultados.

**Decisiones a justificar**
- Métricas principales seleccionadas.
- Qué gráficos se usarán para interpretar errores/aciertos.
- Cómo se comparará el resultado con una referencia básica.

**Entregable de la sección:** tabla de métricas y gráficos de evaluación del modelo.

In [ ]:
# TODO: Calcular métricas de evaluación según el tipo de objetivo
# TODO: Crear gráficos simples de resultados (real vs predicho o matriz de confusión)

## 8. Conclusiones y limitaciones

Finalmente, se resumirán hallazgos clave, limitaciones metodológicas y riesgos del enfoque.
El objetivo es cerrar con una interpretación honesta y accionable.

**Decisiones a justificar**
- Qué hipótesis quedó mejor respaldada.
- Qué límites tienen datos y modelo.
- Qué mejoras serían prioritarias en una siguiente iteración.

**Entregable de la sección:** conclusión ejecutiva y lista de limitaciones/siguientes pasos.

In [ ]:
# TODO: Redactar conclusiones finales basadas en resultados
# TODO: Documentar limitaciones y próximos pasos

## Checklist de ejecución

- [ ] 1. Objetivo y tesis financiera
- [ ] 2. Carga y alineación temporal de datos
- [ ] 3. EDA (exploración de calidad)
- [ ] 4. Limpieza y preprocesado
- [ ] 5. Feature engineering
- [ ] 6. Modelado supervisado
- [ ] 7. Evaluación y visualización
- [ ] 8. Conclusiones y limitaciones

## Notas de reproducibilidad

- Trabajar siempre con rutas relativas del proyecto.
- Evitar data leakage en cualquier transformación.
- Usar split temporal (sin `shuffle`) para train/test.
- Comentar celdas con claridad para trazabilidad académica.